# Streamlit RAG 앱 — 강사용 정답본

원본: `solution/Chapter01/06-rag-streamlit/app.py`  
대응 슬라이드: `99_pdf/Chapter01/day02/2-3-streamlit-rag.pdf`

**학습 목표:** RAG 체인 캐시 → 메시지 표시 → 질문·답변 저장의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** Streamlit 앱에서 질문하면 로컬 PDF 읽기, 임베딩 API, LLM 호출이 발생한다. UI 동작은 원본 .py를 Streamlit으로 실행해 확인한다.



## 흐름을 먼저 읽기

RAG 체인 캐시 → 메시지 표시 → 질문·답변 저장

핵심은 스크립트 재실행과 화면 상태의 관계이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: RAG 체인 캐시 → 메시지 표시 → 질문·답변 저장. session_state의 기록을 LLM도 자동으로 기억하는가? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 노트북 경로 설정

원본은 `__file__`로 PDF 위치를 찾는다. 노트북에서는 정의되지 않을 수 있으므로 원본 코드셀보다 먼저 이 설정 셀을 수동 실행한다. 프로젝트 루트를 찾지 못하면 명시적으로 오류가 난다. 원본 코드는 수정하지 않는다.


In [ ]:
from pathlib import Path
_cwd = Path.cwd().resolve()
_roots = [p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').is_file()]
if not _roots:
    raise FileNotFoundError('pyproject.toml이 있는 프로젝트 폴더에서 노트북을 열어 주세요.')
_project_root = _roots[0]
__file__ = str(_project_root / 'solution/Chapter01/06-rag-streamlit/app.py')
if not Path(__file__).is_file():
    raise FileNotFoundError(f'원본 Python 파일을 찾을 수 없습니다: {__file__}')


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** Streamlit 앱에서 질문하면 로컬 PDF 읽기, 임베딩 API, LLM 호출이 발생한다. UI 동작은 원본 .py를 Streamlit으로 실행해 확인한다.



In [ ]:
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFMinerLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()

# ── RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) ─────────────────────────
@st.cache_resource
def load_rag_chain():
    PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'

    loader = PDFMinerLoader(str(PDF_PATH.resolve()))
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
    chunks = splitter.split_documents(documents)

    embeddings = get_embeddings()
    vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ('system', '당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.\n\ncontext:\n{context}'),
        ('human', '{question}'),
    ])

    def format_docs(docs):
        return '\n\n'.join(doc.page_content for doc in docs)

    chain = (
        {'context': retriever | format_docs, 'question': RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


# ── Streamlit UI ──────────────────────────────────────────────────────────────

st.title('보험 약관 QA 챗봇')
st.caption('메리츠 펫보험 약관을 기반으로 답변합니다.')

# 대화 기록을 session_state에 저장 (새로고침 전까지 유지)
if 'messages' not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.write(msg['content'])

# 사용자 입력
if prompt := st.chat_input('질문을 입력하세요 (예: 보험료는 어떻게 계산되나요?)'):
    # 사용자 메시지 표시
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.write(prompt)

    # AI 응답 (스트리밍)
    with st.chat_message('assistant'):
        rag_chain = load_rag_chain()
        response = st.write_stream(rag_chain.stream(prompt))

    st.session_state.messages.append({'role': 'assistant', 'content': response})


## 결과를 검토하기

- 예상 동작: 앱의 질문 입력에 문서 기반 답변이 스트리밍으로 나타난다.
- 확인 질문: session_state의 기록을 LLM도 자동으로 기억하는가?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
